# 1.1 文字怎麼變成數字？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像查字典：ID 找到一列向量**

ID 只是索引；向量的內容由訓練改變。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/lookup.svg")))

**先想一想：**如果把貓的 ID 改成 99，意思會變嗎？

電腦先處理編號，不直接處理字義。編號只是索引：「貓=0」不表示貓比狗小。反向表讓結果重新變成文字。

**把直覺寫成數學：**一個有限集合到整數的對應；這一步沒有可學參數。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
text = "貓看狗，狗看貓。"
chars = sorted(set(text))
to_id = {c: i for i, c in enumerate(chars)}
ids = [to_id[c] for c in text]
restored = "".join(chars[i] for i in ids)
print(to_id, ids, restored)
assert restored == text

**如何讀結果：**assert 檢查編碼再解碼仍是原文；排序只為讓每次編號可重現。

**只改一件事：**只交換兩個 ID，同時更新反向表，檢查還原是否不變。
